# Task 4 — k-Nearest Neighbours Classifier

Three parts:
1. **Part I** — Iris dataset with scikit-learn `KNeighborsClassifier`
2. **Part II** — Occupancy Detection dataset (pre-split train/test files)
3. **Part III** — Custom kNN implementation using the **chi-squared distance function**

In [2]:
import pandas
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.neighbors import KNeighborsClassifier

---
## Part I — Iris Dataset (sklearn kNN)


### (a) Load the dataset

In [3]:
names = ['sepal-length', 'sepal-width', 'petal-length', 'petal-width', 'class']
dataset = pandas.read_csv('Datasets/data/iris.data', names=names)
dataset.head()

,sepal-length,sepal-width,petal-length,petal-width,class
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


### (b) Print the size of the dataset

In [4]:
print("Shape:", dataset.shape)   # expected [150, 5]

Shape: (150, 5)


### (c) Display the class distribution

In [5]:
print("Class distribution:")
print(dataset.groupby('class').size())

Class distribution:
class
Iris-setosa        50
Iris-versicolor    50
Iris-virginica     50
dtype: int64


### (d) Split into training (80%) and test (20%) sets

In [6]:
array = dataset.values
X = array[:, 0:4].astype(float)
Y = array[:, 4]

t_size = 0.20
seed   = 7

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=t_size, random_state=seed
)

print(f"Training samples : {len(X_train)}")
print(f"Test samples     : {len(X_test)}")

Training samples : 120
Test samples     : 30


### (e) Apply kNN Classifier (default k = 5)

In [7]:
knn = KNeighborsClassifier()        # default n_neighbors=5
knn.fit(X_train, Y_train)
predictions = knn.predict(X_test)

print("Accuracy score:")
print(accuracy_score(Y_test, predictions))

print("\nConfusion matrix:")
print(confusion_matrix(Y_test, predictions))

print("\nClassification report:")
print(classification_report(Y_test, predictions))

Accuracy score:
0.9

Confusion matrix:
[[ 7  0  0]
 [ 0 11  1]
 [ 0  2  9]]

Classification report:
                 precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00         7
Iris-versicolor       0.85      0.92      0.88        12
 Iris-virginica       0.90      0.82      0.86        11

       accuracy                           0.90        30
      macro avg       0.92      0.91      0.91        30
   weighted avg       0.90      0.90      0.90        30



### (f) Vary k from 1 to 10 — print accuracy for each

In [8]:
print("k-sweep (seed=7, test_size=0.20):")
for k in range(1, 11):
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, Y_train)
    acc = accuracy_score(Y_test, model.predict(X_test))
    print(f"  k={k}, Accuracy= {acc}")

k-sweep (seed=7, test_size=0.20):
  k=1, Accuracy= 0.9
  k=2, Accuracy= 0.9333333333333333
  k=3, Accuracy= 0.9
  k=4, Accuracy= 0.9333333333333333
  k=5, Accuracy= 0.9
  k=6, Accuracy= 0.8666666666666667
  k=7, Accuracy= 0.8666666666666667
  k=8, Accuracy= 0.9
  k=9, Accuracy= 0.9
  k=10, Accuracy= 0.9


### (g) Vary seed from 1 to 10 — print accuracy for each

In [9]:
print("Seed sweep (k=5, test_size=0.20):")
for s in range(1, 11):
    Xtr, Xte, Ytr, Yte = train_test_split(X, Y, test_size=0.20, random_state=s)
    model = KNeighborsClassifier(n_neighbors=5)
    model.fit(Xtr, Ytr)
    acc = accuracy_score(Yte, model.predict(Xte))
    print(f"  seed={s}, Accuracy= {acc}")

Seed sweep (k=5, test_size=0.20):
  seed=1, Accuracy= 1.0
  seed=2, Accuracy= 1.0
  seed=3, Accuracy= 0.9666666666666667
  seed=4, Accuracy= 0.9666666666666667
  seed=5, Accuracy= 0.9333333333333333
  seed=6, Accuracy= 0.9666666666666667
  seed=7, Accuracy= 0.9
  seed=8, Accuracy= 0.9
  seed=9, Accuracy= 1.0
  seed=10, Accuracy= 0.9666666666666667


---
## Part II — Occupancy Detection Dataset (sklearn kNN)

Source: http://archive.ics.uci.edu/ml/datasets/Occupancy+Detection+

The dataset comes **pre-split** into training and test files:
- `datatraining.txt` — training data (8 143 rows)
- `datatest.txt`     — test data  (2 665 rows)
- `datatest2.txt`    — second test set (9 752 rows)

**Features used** (Date is ignored): `Temperature`, `Humidity`, `Light`, `CO2`, `HumidityRatio`  
**Target**: `Occupancy` (0 = unoccupied, 1 = occupied)

Steps (d) and (g) from Part I are **not applicable** — the split is already fixed.

### Load the occupancy data

In [10]:
train_occ = pandas.read_csv('Datasets/data/datatraining.txt')
test_occ  = pandas.read_csv('Datasets/data/datatest.txt')
test_occ2 = pandas.read_csv('Datasets/data/datatest2.txt')

print("Training set shape:", train_occ.shape)
print("Test set 1 shape:  ", test_occ.shape)
print("Test set 2 shape:  ", test_occ2.shape)
print("\nColumns:", list(train_occ.columns))
train_occ.head()

Training set shape: (8143, 7)
Test set 1 shape:   (2665, 7)
Test set 2 shape:   (9752, 7)

Columns: ['date', 'Temperature', 'Humidity', 'Light', 'CO2', 'HumidityRatio', 'Occupancy']


,date,Temperature,Humidity,Light,CO2,HumidityRatio,Occupancy
1,2015-02-04 17:51:00,23.18,27.2720,426.0,721.25,0.004793,1
2,2015-02-04 17:51:59,23.15,27.2675,429.5,714.00,0.004783,1
3,2015-02-04 17:53:00,23.15,27.2450,426.0,713.50,0.004779,1
4,2015-02-04 17:54:00,23.15,27.2000,426.0,708.25,0.004772,1
5,2015-02-04 17:55:00,23.10,27.2000,426.0,704.50,0.004757,1


### Class distribution

In [11]:
print("Training set — class distribution:")
print(train_occ.groupby('Occupancy').size())

print("\nTest set 1 — class distribution:")
print(test_occ.groupby('Occupancy').size())

Training set — class distribution:
Occupancy
0    6414
1    1729
dtype: int64

Test set 1 — class distribution:
Occupancy
0    1693
1     972
dtype: int64


### Prepare feature matrices (drop Date)

In [12]:
features = ['Temperature', 'Humidity', 'Light', 'CO2', 'HumidityRatio']

X_train_occ = train_occ[features].values
Y_train_occ = train_occ['Occupancy'].values

X_test_occ  = test_occ[features].values
Y_test_occ  = test_occ['Occupancy'].values

X_test_occ2 = test_occ2[features].values
Y_test_occ2 = test_occ2['Occupancy'].values

print(f"X_train: {X_train_occ.shape}  |  X_test1: {X_test_occ.shape}  |  X_test2: {X_test_occ2.shape}")

X_train: (8143, 5)  |  X_test1: (2665, 5)  |  X_test2: (9752, 5)


### Apply kNN Classifier (default k = 5)

In [13]:
knn_occ = KNeighborsClassifier(n_neighbors=5)
knn_occ.fit(X_train_occ, Y_train_occ)
preds_occ = knn_occ.predict(X_test_occ)

print("=== Test set 1 ===")
print("Accuracy :", accuracy_score(Y_test_occ, preds_occ))
print("\nConfusion matrix:")
print(confusion_matrix(Y_test_occ, preds_occ))
print("\nClassification report:")
print(classification_report(Y_test_occ, preds_occ))

=== Test set 1 ===
Accuracy : 0.9425891181988743

Confusion matrix:
[[1645   48]
 [ 105  867]]

Classification report:
              precision    recall  f1-score   support

           0       0.94      0.97      0.96      1693
           1       0.95      0.89      0.92       972

    accuracy                           0.94      2665
   macro avg       0.94      0.93      0.94      2665
weighted avg       0.94      0.94      0.94      2665



In [14]:
preds_occ2 = knn_occ.predict(X_test_occ2)

print("=== Test set 2 ===")
print("Accuracy :", accuracy_score(Y_test_occ2, preds_occ2))
print("\nConfusion matrix:")
print(confusion_matrix(Y_test_occ2, preds_occ2))
print("\nClassification report:")
print(classification_report(Y_test_occ2, preds_occ2))

=== Test set 2 ===
Accuracy : 0.9621616078753076

Confusion matrix:
[[7385  318]
 [  51 1998]]

Classification report:
              precision    recall  f1-score   support

           0       0.99      0.96      0.98      7703
           1       0.86      0.98      0.92      2049

    accuracy                           0.96      9752
   macro avg       0.93      0.97      0.95      9752
weighted avg       0.97      0.96      0.96      9752



### Vary k from 1 to 10

In [15]:
print("k-sweep — Occupancy Detection (Test set 1):")
for k in range(1, 11):
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_occ, Y_train_occ)
    acc1 = accuracy_score(Y_test_occ,  model.predict(X_test_occ))
    acc2 = accuracy_score(Y_test_occ2, model.predict(X_test_occ2))
    print(f"  k={k:2d},  Test1 Accuracy= {acc1:.4f},  Test2 Accuracy= {acc2:.4f}")

k-sweep — Occupancy Detection (Test set 1):
  k= 1,  Test1 Accuracy= 0.9366,  Test2 Accuracy= 0.9503
  k= 2,  Test1 Accuracy= 0.9231,  Test2 Accuracy= 0.9525
  k= 3,  Test1 Accuracy= 0.9351,  Test2 Accuracy= 0.9580
  k= 4,  Test1 Accuracy= 0.9276,  Test2 Accuracy= 0.9540
  k= 5,  Test1 Accuracy= 0.9426,  Test2 Accuracy= 0.9622
  k= 6,  Test1 Accuracy= 0.9325,  Test2 Accuracy= 0.9629
  k= 7,  Test1 Accuracy= 0.9610,  Test2 Accuracy= 0.9649
  k= 8,  Test1 Accuracy= 0.9550,  Test2 Accuracy= 0.9659
  k= 9,  Test1 Accuracy= 0.9617,  Test2 Accuracy= 0.9656
  k=10,  Test1 Accuracy= 0.9598,  Test2 Accuracy= 0.9656


---
## Part III — Custom kNN with Chi-Squared Distance


### Chi-squared distance function

In [16]:
def chi2_distance(a, b):
    """
    Chi-squared distance between two 1-D numeric arrays.
    d(a, b) = sum( (a_i - b_i)^2 / (a_i + b_i) )   for each i where (a_i+b_i) != 0
    """
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    denom = a + b
    with np.errstate(invalid='ignore', divide='ignore'):
        terms = np.where(denom != 0.0, (a - b) ** 2 / denom, 0.0)
    return float(np.sum(terms))

# Quick sanity check
print("chi2([1,2,3], [1,2,3]) =", chi2_distance([1,2,3], [1,2,3]))   # identical → 0
print("chi2([1,0,1], [0,0,2]) =", chi2_distance([1,0,1], [0,0,2]))

chi2([1,2,3], [1,2,3]) = 0.0
chi2([1,0,1], [0,0,2]) = 1.3333333333333333


### Custom kNN classifier

In [23]:
def knn_chi2_predict_fast(X_train, Y_train, X_test, k=5):
    Y_train = np.asarray(Y_train)
    predictions = []
    
    for x in X_test:
        # Vectorized chi2 distance calculation
        denom = X_train + x
        with np.errstate(invalid='ignore', divide='ignore'):
            terms = np.where(denom != 0.0, (X_train - x) ** 2 / denom, 0.0)
        distances = np.sum(terms, axis=1)
        
        nn_indices = np.argsort(distances)[:k]
        nn_labels = Y_train[nn_indices]
        labels, counts = np.unique(nn_labels, return_counts=True)
        predictions.append(labels[np.argmax(counts)])
    
    return np.array(predictions)

### Part III-I — Iris dataset (custom kNN)

In [24]:
print("Custom kNN (chi-squared) on Iris — full report (k=5, seed=7):")
preds_iris_custom = knn_chi2_predict(X_train, Y_train, X_test, k=5)

print("Accuracy :", accuracy_score(Y_test, preds_iris_custom))
print("\nConfusion matrix:")
print(confusion_matrix(Y_test, preds_iris_custom))
print("\nClassification report:")
print(classification_report(Y_test, preds_iris_custom))

Custom kNN (chi-squared) on Iris — full report (k=5, seed=7):
Accuracy : 0.8666666666666667

Confusion matrix:
[[ 7  0  0]
 [ 0 10  2]
 [ 0  2  9]]

Classification report:
                 precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00         7
Iris-versicolor       0.83      0.83      0.83        12
 Iris-virginica       0.82      0.82      0.82        11

       accuracy                           0.87        30
      macro avg       0.88      0.88      0.88        30
   weighted avg       0.87      0.87      0.87        30



In [25]:
print("k-sweep — Iris (custom chi-squared kNN, seed=7):")
for k in range(1, 11):
    preds = knn_chi2_predict(X_train, Y_train, X_test, k=k)
    acc   = accuracy_score(Y_test, preds)
    print(f"  k={k}, Accuracy= {acc}")

k-sweep — Iris (custom chi-squared kNN, seed=7):
  k=1, Accuracy= 0.9
  k=2, Accuracy= 0.9333333333333333
  k=3, Accuracy= 0.9
  k=4, Accuracy= 0.8666666666666667
  k=5, Accuracy= 0.8666666666666667
  k=6, Accuracy= 0.8666666666666667
  k=7, Accuracy= 0.8666666666666667
  k=8, Accuracy= 0.8666666666666667
  k=9, Accuracy= 0.9
  k=10, Accuracy= 0.9


In [26]:
print("Seed sweep — Iris (custom chi-squared kNN, k=5):")
for s in range(1, 11):
    Xtr, Xte, Ytr, Yte = train_test_split(X, Y, test_size=0.20, random_state=s)
    preds = knn_chi2_predict(Xtr, Ytr, Xte, k=5)
    acc   = accuracy_score(Yte, preds)
    print(f"  seed={s}, Accuracy= {acc}")

Seed sweep — Iris (custom chi-squared kNN, k=5):
  seed=1, Accuracy= 0.9666666666666667
  seed=2, Accuracy= 0.9666666666666667
  seed=3, Accuracy= 0.9666666666666667
  seed=4, Accuracy= 0.9333333333333333
  seed=5, Accuracy= 0.9333333333333333
  seed=6, Accuracy= 1.0
  seed=7, Accuracy= 0.8666666666666667
  seed=8, Accuracy= 0.9
  seed=9, Accuracy= 1.0
  seed=10, Accuracy= 0.9666666666666667


### Part III-II — Occupancy Detection (custom kNN)

In [27]:
print("Custom kNN (chi-squared) on Occupancy — Test set 1 (k=5):")
preds_occ_custom = knn_chi2_predict(X_train_occ, Y_train_occ, X_test_occ, k=5)

print("Accuracy :", accuracy_score(Y_test_occ, preds_occ_custom))
print("\nConfusion matrix:")
print(confusion_matrix(Y_test_occ, preds_occ_custom))
print("\nClassification report:")
print(classification_report(Y_test_occ, preds_occ_custom))

Custom kNN (chi-squared) on Occupancy — Test set 1 (k=5):
Accuracy : 0.9332082551594747

Confusion matrix:
[[1645   48]
 [ 130  842]]

Classification report:
              precision    recall  f1-score   support

           0       0.93      0.97      0.95      1693
           1       0.95      0.87      0.90       972

    accuracy                           0.93      2665
   macro avg       0.94      0.92      0.93      2665
weighted avg       0.93      0.93      0.93      2665



In [28]:
print("Custom kNN (chi-squared) on Occupancy — Test set 2 (k=5):")
preds_occ2_custom = knn_chi2_predict(X_train_occ, Y_train_occ, X_test_occ2, k=5)

print("Accuracy :", accuracy_score(Y_test_occ2, preds_occ2_custom))
print("\nConfusion matrix:")
print(confusion_matrix(Y_test_occ2, preds_occ2_custom))
print("\nClassification report:")
print(classification_report(Y_test_occ2, preds_occ2_custom))

Custom kNN (chi-squared) on Occupancy — Test set 2 (k=5):
Accuracy : 0.9573420836751435

Confusion matrix:
[[7403  300]
 [ 116 1933]]

Classification report:
              precision    recall  f1-score   support

           0       0.98      0.96      0.97      7703
           1       0.87      0.94      0.90      2049

    accuracy                           0.96      9752
   macro avg       0.93      0.95      0.94      9752
weighted avg       0.96      0.96      0.96      9752



### Had to be stopped (was taking too much time)

In [29]:
print("k-sweep — Occupancy Detection (custom chi-squared kNN):")
for k in range(1, 11):
    p1 = knn_chi2_predict(X_train_occ, Y_train_occ, X_test_occ,  k=k)
    p2 = knn_chi2_predict(X_train_occ, Y_train_occ, X_test_occ2, k=k)
    a1 = accuracy_score(Y_test_occ,  p1)
    a2 = accuracy_score(Y_test_occ2, p2)
    print(f"  k={k:2d},  Test1 Accuracy= {a1:.4f},  Test2 Accuracy= {a2:.4f}")

k-sweep — Occupancy Detection (custom chi-squared kNN):
  k= 1,  Test1 Accuracy= 0.9212,  Test2 Accuracy= 0.9505


KeyboardInterrupt: 